## Malaria Detection Using CNN

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow_datasets as tfds

In [ ]:
dataset, dataset_info = tfds.load('malaria',with_info=True)

In [ ]:
dataset, dataset_info = tfds.load(
                                  'malaria',
                                  with_info=True,
                                  # split=['train[:80%]','train[80%:90%]','train[90%:]'],
                                  shuffle_files=True)

In [ ]:
Dataset_size=len(dataset['train'])

In [ ]:
def split(ds, train_ratio, val_ratio):
    n = len(ds)
    train_n = int(train_ratio * n)
    val_n = int(val_ratio * n)

    train_ds = ds.take(train_n)
    rest = ds.skip(train_n)
    val_ds = rest.take(val_n)
    test_ds = rest.skip(val_n)
    return train_ds, val_ds, test_ds

train_dataset, val_dataset, test_dataset = split(dataset['train'], 0.8, 0.1)

print(len(train_dataset), len(val_dataset), len(test_dataset))

## Data Visualization

In [ ]:
def getLabel(label):
  if label == 0:
    return "Parasitized"
  else:
    return "Uninfected"

In [ ]:
plt.figure(figsize=(12,12))
for i,sample in enumerate(train_dataset.take(16)):
  plt.subplot(4,4,i+1)
  plt.imshow(sample['image'])
  plt.title(getLabel(sample['label'].numpy()))

## Rescaling The Images

In [ ]:
IM_size=224

In [ ]:
def resize_rescale(input):
  image = tf.image.resize(input['image'],(IM_size,IM_size))/255.
  return image,input['label']

In [ ]:
train_dataset=train_dataset.map(resize_rescale).shuffle(buffer_size=8, reshuffle_each_itearation=True).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset=val_dataset.map(resize_rescale).shuffle(buffer_size=8, reshuffle_each_itearation=True).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset=test_dataset.map(resize_rescale).shuffle(buffer_size=8, reshuffle_each_itearation=True).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
for image,label in test_dataset.take(1):
  print(image.shape)

In [ ]:
np.unique(image)

## Model Creation

In [ ]:
from tensorflow.python.distribute.cross_device_ops import kernels
model = tf.keras.Sequential([
    layers.Input(shape=(IM_size,IM_size,3)),

    ## Conv2d Layer
    layers.Conv2D(filters=6,kernel_size=5,strides=1,padding='valid',activation='sigmoid'),
    layers.MaxPool2D(pool_size=(2,2),strides=None),

    ## Conv2d Layer
    layers.Conv2D(filters=16,kernel_size=5,strides=1,padding='valid',activation='sigmoid'),
    layers.MaxPool2D(pool_size=(2,2),strides=None),

    ## Flatten Layer
    layers.Flatten(),

    ## Dense Layer
    Dense(100,activation='sigmoid'),
    Dense(10,activation='sigmoid'),
    Dense(2,activation='sigmoid')
])
model.summary()



In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.keras.losses.BinaryCrossentropy(),
)

In [ ]:
model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=100,
    verbose=1
)